# 🎯 Support Vector Machine (SVM) — Wine Quality Prediction

---

## 🎯 What Will You Learn?

- Understand the SVM intuition with a simple visual analogy
- Learn about **support vectors**, **hyperplanes**, and **margins**
- Understand the difference between **kernels** (Linear, RBF, Polynomial)
- Learn the effect of hyperparameters **C** and **gamma**
- Compare SVM with Random Forest on the same data

---

## 📖 Table of Contents
1. What is SVM?
2. Loading and Preparing Wine Data
3. SVM with Linear Kernel
4. SVM with RBF Kernel
5. Understanding Hyperparameters (C and gamma)
6. Finding Best Parameters
7. Final Evaluation
8. SVM vs Random Forest Comparison
9. Key Takeaways


---
## 1. 🎯 What is Support Vector Machine (SVM)?

### The "Best Boundary" Analogy 📏

Imagine you have two groups of points on a table:
- 🔴 Red dots = Bad wines (quality < 7)
- 🟢 Green dots = Good wines (quality ≥ 7)

You want to draw a LINE that separates them. But there are MANY possible lines!

```
        ❌ Line A    ✅ Line B (Best!)    ❌ Line C
           |              |                |
  🟢 🟢   |   🟢 🟢   |              |   🟢 🟢
  🔴 🔴   |   🔴 🔴   |              |   🔴 🔴
```

**SVM finds the line (or plane) with the MAXIMUM MARGIN between the two groups!**

### Key Terms:

**Support Vectors:** The data points closest to the dividing line — these are the "important" points that define the boundary.

**Hyperplane:** The dividing line (in 2D) or plane (in 3D) or boundary (in higher dimensions).

**Margin:** The distance between the hyperplane and the nearest support vectors. **SVM maximizes this margin!**

```
         Support        Support
         Vector         Vector
            ↓              ↓
  🟢   🟢 [🟢] ←margin→ [🔴] 🔴   🔴
            |______________|_____
                        Hyperplane
```

### Why Maximum Margin?
- A line very close to the points is fragile — small changes cause misclassification
- A line with maximum margin is the most confident, robust separator

### The Kernel Trick — Handling Non-Linear Data

What if the data can't be separated by a straight line?

```
Original 2D data (not linearly separable):
  🔴 🔴 🟢 🟢 🔴 🔴
  🟢 🔴 🔴 🔴 🔴 🟢
  
After Kernel Transform (now separable in higher dimension!):
  A curved boundary in 2D → A flat boundary in 3D
```

**Kernel Types:**
- **Linear:** Straight line/plane (for linearly separable data)
- **RBF (Radial Basis Function):** Circular/curved boundaries (most common)
- **Polynomial:** Uses polynomial curves


In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries imported!")

---
## 2. 📂 Loading and Preparing Wine Data

(Same preprocessing as the Random Forest notebook)

In [ ]:
# Load and prepare data
df = pd.read_csv('data/WineQT.csv').drop('Id', axis=1)

# Create binary target: Good (≥7) vs Not Good (<7)
df['good_wine'] = (df['quality'] >= 7).astype(int)

X = df.drop(['quality', 'good_wine'], axis=1)
y = df['good_wine']

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Target distribution: {y.value_counts().to_dict()}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ⚠️ SCALING IS CRITICAL FOR SVM!
# SVM measures distances between points
# Without scaling, features with large values dominate completely
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data scaled and split!")
print(f"   Training: {len(X_train)} | Testing: {len(X_test)}")
print()
print("Before scaling:")
print(f"  'fixed acidity' range: {X_train['fixed acidity'].min():.1f} to {X_train['fixed acidity'].max():.1f}")
print(f"  'residual sugar' range: {X_train['residual sugar'].min():.1f} to {X_train['residual sugar'].max():.1f}")
print()
print("After scaling (both centered near 0):")
print(f"  'fixed acidity' range: {X_train_scaled[:,0].min():.2f} to {X_train_scaled[:,0].max():.2f}")
print(f"  'residual sugar' range: {X_train_scaled[:,3].min():.2f} to {X_train_scaled[:,3].max():.2f}")

### 🎨 Visualizing SVM Decision Boundary (2D Example)

Let's first see how SVM works in 2D using just 2 features:

In [ ]:
# Demonstrate SVM boundary in 2D using 2 most important features
from sklearn.preprocessing import StandardScaler as SS

# Use only 2 features for visualization
X_2d = df[['alcohol', 'volatile acidity']].values
y_2d = df['good_wine'].values

X_2d_train, X_2d_test, y_2d_train, y_2d_test = train_test_split(
    X_2d, y_2d, test_size=0.2, random_state=42, stratify=y_2d
)

scaler_2d = SS()
X_2d_train_s = scaler_2d.fit_transform(X_2d_train)
X_2d_test_s = scaler_2d.transform(X_2d_test)

# Train SVMs with different kernels
kernels = ['linear', 'rbf', 'poly']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, kernel in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1, random_state=42)
    svm.fit(X_2d_train_s, y_2d_train)
    
    # Create decision boundary
    xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
    
    # Plot data points
    scatter = ax.scatter(X_2d_train_s[y_2d_train==0, 0], X_2d_train_s[y_2d_train==0, 1],
                         c='#e74c3c', alpha=0.6, s=20, label='Not Good')
    ax.scatter(X_2d_train_s[y_2d_train==1, 0], X_2d_train_s[y_2d_train==1, 1],
               c='#2ecc71', alpha=0.6, s=20, label='Good')
    
    # Highlight support vectors
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=80, facecolors='none', 
               edgecolors='black', linewidths=1.5, label=f'Support Vectors ({len(sv)})')
    
    acc = accuracy_score(y_2d_test, svm.predict(X_2d_test_s))
    ax.set_title(f'SVM — {kernel.upper()} Kernel\nAccuracy: {acc*100:.1f}%', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Alcohol (scaled)', fontsize=11)
    ax.set_ylabel('Volatile Acidity (scaled)', fontsize=11)
    ax.legend(fontsize=9, loc='upper right')

plt.suptitle('SVM Decision Boundaries with Different Kernels (2D View)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('svm_boundaries_2d.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Red zone = predicted Not Good | Green zone = predicted Good")
print("   The black circles are the SUPPORT VECTORS (most important data points!)")

---
## 3. 🔵 SVM with Linear Kernel

In [ ]:
# SVM with Linear Kernel (full dataset, all features)
# Linear = finds a straight hyperplane to separate the classes
print("🚀 Training SVM with Linear Kernel (all 11 features)...")
svm_linear = SVC(
    kernel='linear',       # Straight-line boundary
    C=1.0,                 # Regularization (explained later)
    random_state=42,
    probability=True       # Needed to get probability scores for ROC curve
)
svm_linear.fit(X_train_scaled, y_train)

y_pred_lin = svm_linear.predict(X_test_scaled)
y_prob_lin = svm_linear.predict_proba(X_test_scaled)

acc_lin = accuracy_score(y_test, y_pred_lin)
auc_lin = roc_auc_score(y_test, y_prob_lin[:, 1])

print(f"\nLinear SVM Results:")
print(f"  Accuracy: {acc_lin*100:.2f}%")
print(f"  ROC-AUC:  {auc_lin:.4f}")
print(f"  Number of Support Vectors: {sum(svm_linear.n_support_)}")

---
## 4. 🌊 SVM with RBF Kernel (Recommended)

**RBF = Radial Basis Function** — the most popular and versatile SVM kernel.

Instead of a straight line, RBF creates **curved, circular boundaries** around clusters of data.

In [ ]:
# SVM with RBF Kernel
print("🚀 Training SVM with RBF Kernel...")
svm_rbf = SVC(
    kernel='rbf',          # Radial Basis Function — curved boundary
    C=1.0,                 # Regularization parameter
    gamma='scale',         # Kernel coefficient ('scale' = 1/(n_features * X.var()))
    random_state=42,
    probability=True
)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
y_prob_rbf = svm_rbf.predict_proba(X_test_scaled)

acc_rbf = accuracy_score(y_test, y_pred_rbf)
auc_rbf = roc_auc_score(y_test, y_prob_rbf[:, 1])

print(f"\nRBF SVM Results:")
print(f"  Accuracy: {acc_rbf*100:.2f}%")
print(f"  ROC-AUC:  {auc_rbf:.4f}")
print(f"  Number of Support Vectors: {sum(svm_rbf.n_support_)}")

---
## 5. 🎛️ Understanding Hyperparameters: C and gamma

### The C Parameter (Regularization)

**C controls the trade-off between having a wide margin vs. correctly classifying all training points.**

```
Small C (e.g., 0.01):
  → Wide margin, allows some misclassifications
  → More general (less overfit)
  → "I care more about the margin than getting every point right"
  
Large C (e.g., 1000):
  → Narrow margin, tries to classify every point correctly
  → Can overfit to training data
  → "I really don't want to misclassify even one point"
```

### The gamma Parameter (RBF only)

**Gamma controls how far each training example's influence reaches.**

```
Small gamma (e.g., 0.001):
  → Each point has far reach → smooth, simple boundary
  → Might underfit (misses complex patterns)
  
Large gamma (e.g., 100):
  → Each point has close reach → complex, wiggly boundary  
  → Might overfit (too specific to training data)
```

In [ ]:
# Effect of C on accuracy
print("Effect of C parameter (gamma='scale', RBF kernel):")
print(f"{'C value':<12} {'Train Acc':<12} {'Test Acc':<12} {'AUC':<10} {'Verdict'}")
print("-" * 60)

for C in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    svm = SVC(kernel='rbf', C=C, gamma='scale', probability=True, random_state=42)
    svm.fit(X_train_scaled, y_train)
    tr_acc = accuracy_score(y_train, svm.predict(X_train_scaled))
    te_acc = accuracy_score(y_test, svm.predict(X_test_scaled))
    auc = roc_auc_score(y_test, svm.predict_proba(X_test_scaled)[:,1])
    
    gap = tr_acc - te_acc
    if gap > 0.1:
        verdict = "⚠️  Overfit"
    elif te_acc > 0.88:
        verdict = "✅ Excellent"
    elif te_acc > 0.85:
        verdict = "✅ Good"
    else:
        verdict = "🟡 OK"
    print(f"{C:<12} {tr_acc*100:.1f}%      {te_acc*100:.1f}%      {auc:.4f}    {verdict}")

In [ ]:
# Visualize C vs accuracy
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_accs_c = []
test_accs_c = []

for C in C_values:
    svm = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    svm.fit(X_train_scaled, y_train)
    train_accs_c.append(accuracy_score(y_train, svm.predict(X_train_scaled))*100)
    test_accs_c.append(accuracy_score(y_test, svm.predict(X_test_scaled))*100)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(len(C_values)), train_accs_c, 'b-o', linewidth=2, markersize=8,
        label='Training Accuracy', color='#3498db')
ax.plot(range(len(C_values)), test_accs_c, 'r-o', linewidth=2, markersize=8,
        label='Test Accuracy', color='#e74c3c')
ax.set_xticks(range(len(C_values)))
ax.set_xticklabels([str(c) for c in C_values])
ax.set_xlabel('C value', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Effect of C Parameter on SVM Accuracy\n(Small C = underfitting, Large C = overfitting)', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('svm_C_effect.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 6. 🔍 Finding Best Parameters with Grid Search

In [ ]:
# Grid Search: Try all combinations of C and gamma
# This is like systematically testing every possible setting
print("🔍 Grid Search — finding best C and gamma...")
print("(This may take 1-2 minutes)\n")

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.001, 0.01, 0.1]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42),
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring='roc_auc',   # Optimize for AUC
    n_jobs=-1            # Use all CPU cores
)

grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC:     {grid_search.best_score_:.4f}")

In [ ]:
# Visualize Grid Search results
results_grid = pd.DataFrame(grid_search.cv_results_)

# Pivot table: C vs gamma → mean test score
for gamma_val in ['scale', 0.001, 0.01, 0.1]:
    subset = results_grid[results_grid['param_gamma'] == gamma_val]
    mean_scores = subset.groupby('param_C')['mean_test_score'].mean()
    
# Create heatmap
c_vals = [0.1, 1, 10, 100]
g_vals = ['scale', '0.001', '0.01', '0.1']
score_matrix = np.zeros((len(g_vals), len(c_vals)))

for i, C in enumerate(c_vals):
    for j, g in enumerate(['scale', 0.001, 0.01, 0.1]):
        mask = (results_grid['param_C'] == C) & (results_grid['param_gamma'] == g)
        if mask.sum() > 0:
            score_matrix[j, i] = results_grid[mask]['mean_test_score'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(score_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=[str(c) for c in c_vals],
            yticklabels=g_vals, ax=ax,
            linewidths=0.5, annot_kws={'size': 12})
ax.set_xlabel('C (Regularization)', fontsize=12)
ax.set_ylabel('Gamma', fontsize=12)
ax.set_title('Grid Search: ROC-AUC for Different C and Gamma Values', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('svm_grid_search.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"\n✅ Best combination: C={grid_search.best_params_['C']}, gamma={grid_search.best_params_['gamma']}")

---
## 7. 📏 Final Evaluation

In [ ]:
# Train the best SVM model
best_svm = grid_search.best_estimator_

y_pred_best = best_svm.predict(X_test_scaled)
y_prob_best = best_svm.predict_proba(X_test_scaled)

acc_best = accuracy_score(y_test, y_pred_best)
auc_best = roc_auc_score(y_test, y_prob_best[:, 1])

print(f"Best SVM (C={grid_search.best_params_['C']}, gamma={grid_search.best_params_['gamma']}, kernel=RBF):")
print(f"  Accuracy: {acc_best*100:.2f}%")
print(f"  ROC-AUC:  {auc_best:.4f}")
print()
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Not Good (<7)', 'Good (≥7)']))

In [ ]:
# Confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Not Good', 'Pred: Good'],
            yticklabels=['Actual: Not Good', 'Actual: Good'],
            linewidths=2, linecolor='white', annot_kws={'size': 18, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix — Best SVM', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC curves for all SVM variants
for (model, color, label) in [
    (svm_linear, '#3498db', f'Linear (AUC={auc_lin:.3f})'),
    (svm_rbf, '#e67e22', f'RBF default (AUC={auc_rbf:.3f})'),
    (best_svm, '#e74c3c', f'RBF tuned (AUC={auc_best:.3f})')
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_scaled)[:, 1])
    axes[1].plot(fpr, tpr, lw=2, color=color, label=label)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — SVM Variants', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('svm_final_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 8. ⚖️ SVM vs Random Forest Comparison

In [ ]:
# Train Random Forest for comparison
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)  # No scaling needed for RF
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

# Comprehensive comparison
comparison = pd.DataFrame({
    'Model': ['SVM (Linear)', 'SVM (RBF Default)', 'SVM (RBF Tuned)', 'Random Forest'],
    'Accuracy (%)': [f'{acc_lin*100:.2f}', f'{acc_rbf*100:.2f}', 
                     f'{acc_best*100:.2f}', f'{acc_rf*100:.2f}'],
    'ROC-AUC': [f'{auc_lin:.4f}', f'{auc_rbf:.4f}', 
                f'{auc_best:.4f}', f'{auc_rf:.4f}'],
    'Needs Scaling': ['Yes', 'Yes', 'Yes', 'No'],
    'Interpretable': ['Somewhat', 'No', 'No', 'Feature Importance'],
    'Speed': ['Medium', 'Medium', 'Medium', 'Faster']
})

print("Model Comparison — Wine Quality Prediction:")
print(comparison.to_string(index=False))

In [ ]:
# Final comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy comparison bar chart
model_names = ['SVM\nLinear', 'SVM\nRBF', 'SVM\nRBF Tuned', 'Random\nForest']
accs = [acc_lin*100, acc_rbf*100, acc_best*100, acc_rf*100]
aucs = [auc_lin, auc_rbf, auc_best, auc_rf]
colors = ['#3498db', '#e67e22', '#e74c3c', '#27ae60']

bars = axes[0].bar(model_names, accs, color=colors, edgecolor='black', width=0.6)
axes[0].set_ylim([75, 100])
axes[0].set_ylabel('Test Accuracy (%)', fontsize=12)
axes[0].set_title('Accuracy Comparison: SVM vs Random Forest', fontsize=13, fontweight='bold')
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# ROC curve comparison
for (model, is_scaled, color, label, prob) in [
    (svm_linear, True, '#3498db', f'SVM Linear (AUC={auc_lin:.3f})', y_prob_lin[:, 1]),
    (svm_rbf, True, '#e67e22', f'SVM RBF (AUC={auc_rbf:.3f})', y_prob_rbf[:, 1]),
    (best_svm, True, '#e74c3c', f'SVM Tuned (AUC={auc_best:.3f})', y_prob_best[:, 1]),
    (rf, False, '#27ae60', f'Random Forest (AUC={auc_rf:.3f})', y_prob_rf)
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[1].plot(fpr, tpr, lw=2, color=color, label=label)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('svm_vs_rf.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 9. 🎓 Key Takeaways

### SVM Summary:
- Finds the **maximum margin hyperplane** to separate classes
- **Support vectors** are the data points that define the boundary
- **Kernels** allow SVM to handle non-linear data by transforming feature space
- **Scaling is essential** — SVM is very sensitive to feature magnitudes

### Understanding the Parameters:

| Parameter | Low Value | High Value |
|-----------|-----------|------------|
| **C** | Wide margin, some misclassifications (underfitting) | Narrow margin, all correct (overfitting) |
| **gamma** | Far-reaching influence, smooth boundary | Close influence, complex boundary |

### When to Use SVM?
| ✅ SVM Works Well | ⚠️ Avoid SVM When |
|---|---|
| Clear margin between classes | Very large datasets (slow!) |
| High-dimensional data (text, images) | Very noisy data |
| Small to medium datasets | Need probability outputs directly |
| Non-linear data (use RBF kernel) | Need to interpret the model |

### SVM vs Random Forest:
| | SVM | Random Forest |
|---|---|---|
| **Training Speed** | Slower (O(n²) to O(n³)) | Faster |
| **Prediction Speed** | Fast | Moderate |
| **Scaling Required** | Yes (critical!) | No |
| **Handles Non-linearity** | Yes (kernels) | Yes (naturally) |
| **Interpretability** | Low | Medium (feature importance) |
| **Works With Noisy Data** | Less robust | More robust |


In [ ]:
# Cross validation for final assessment
cv_svm = cross_val_score(best_svm, 
    np.vstack([X_train_scaled, X_test_scaled]),
    np.concatenate([y_train, y_test]),
    cv=5, scoring='accuracy', n_jobs=-1)

print("=" * 55)
print("           SVM SUMMARY")
print("=" * 55)
print(f"  Dataset:         Wine Quality (binary)")
print(f"  Total Wines:     {len(df)}")
print(f"  Best Kernel:     RBF")
print(f"  Best C:          {grid_search.best_params_['C']}")
print(f"  Best Gamma:      {grid_search.best_params_['gamma']}")
print(f"  Accuracy:        {acc_best*100:.2f}%")
print(f"  ROC-AUC:         {auc_best:.4f}")
print(f"  CV Accuracy:     {cv_svm.mean()*100:.2f}% ± {cv_svm.std()*100:.2f}%")
print("=" * 55)
print("Next: Model Evaluation Metrics — deep dive into measuring quality!")